# 1.0 BUSINESS UNDERSTANDING

## 1.1 Business Overview
Modern streaming platforms host thousands of movies, often leaving users overwhelmed by endless choices trapping users into endless scrolling cycles.
Recommendation systems can solve this problem by personalizing the discovery process. Instead of forcing users to search through thousands of movies, the system automatically finds matches for them. By analyzing movie details like plot summaries and genres the system calculates how similar movies are to each other. This stops users from feeling overwhelmed by too many choices and helps them find something to watch instantly.

In this project, a content-based movie recommendation system will be built that makes discovering of new movie faster, easier, and matched to what the user wants to watch.

## 1.2 Problem Statement
With thousands of titles available on modern streaming platforms, users are often overwhelmed by choice, trapping them in endless scrolling cycles. Manually searching through massive catalogs is tedious and frustrating, which ultimately reduces user satisfaction and engagement.
This project addresses this issue by developing a content-based movie recommendation system. By leveraging textual similarity techniques to analyze key movie features, the system automatically identifies and suggests closely related titles turning endless searching into instant, tailored discovery.

## 1.3 Project Objectives

- To preprocess and transform textual movie data into meaningful representation for analysis
- To transform movie information into meaningful numerical representations
- To apply Natural Language Processing (NLP) techniques such as tokenization, stemming, vectorization
- To measure similarity between movies using cosine similarity and generate accurate movie recommendations 
- To build a content-based movie recommendation system and improve movie discovery and user experience through personalized recommendations

## 1.4 Research Questions

- How can textual metadata be used to recommend similar movies and TV shows?
- Which NLP techniques are most effective for content-based recommendation?
- How well does TF-IDF vectorization capture content similarity?
- Can cosine similarity effectively identify related movie content?

## 1.5 Success Criteria

The project will be considered successful if:

- Similar movies are grouped effectively
- Recommendations are generated efficiently
- The recommendation results improve content discovery


# 2.0 DATA UNDERSTANDING

## 2.1 Data Source

TMDB movie data is used as the primary data:
- movies.csv
- credits.csv

These datasets contain movie metadata such as:
- Movie titles
- Genres
- Keywords
- Cast information
- Crew information
- Movie overviews (descriptions)
- Popularity and rating metrics


## 2.2 Load Datasets

In [ ]:
import pandas as pd
movies = pd.read_csv("./data/tmdb_5000_movies.csv")
# Displays the first five rows of the movies dataset.
movies.head()

In [ ]:
credits = pd.read_csv("./data/tmdb_5000_credits.csv")
# Displays the first five rows of the credits dataset.
credits.head()

In [ ]:
# check for present columns
movies.columns


In [ ]:
# check for present columns
credits.columns

In [ ]:
# Displays column names, data types, and missing values in the movies dataset.
movies.info()

In [ ]:
# Displays the structure of the credits dataset.
credits.info()

In [ ]:
# check for missing values in movies dataset
movies.isnull().sum().sort_values(ascending=False)

In [ ]:
# Check for missing values in the credits dataset.
credits.isnull().sum()

In [ ]:
# Check for duplicates
movies.duplicated().sum()
credits.duplicated().sum()

In [ ]:
# statistical summary for numerical columns.
movies.describe()

- The movies dataset contains 20 columns, while the credits dataset contains 4 features.
- Most columns contain complete data with very few missing values. The homepage column contains many missing values and may not be useful for recommendation modeling.
- Columns such as overview, genres, keywords, cast, and crew appear highly relevant for generating movie similarities.
- The datasets contain both numerical and textual data types.
- No duplicated rows detected

# 3.0 DATA PREPARATION

In this phase, the datasets will be cleaned and transformed into a format suitable for building the recommendation system.

The main tasks include:
- merging datasets
- selecting relevant features/columns
- handling missing values
- preprocessing textual data
- creating tags for similarity analysis

cleaning data
handling missing values
removing duplicates
feature engineering
text preprocessing
merging datasets



## 3.1 Dataset Merging

In [ ]:
# Merge the movies & credits Datasets

credits.drop('title', axis=1, inplace=True)

movies = movies.merge(credits, left_on='id', right_on='movie_id')
movies.head()

In [ ]:
movies.columns

## 3.2 Feature Selection

In [ ]:
# selected only relevant columns
movies = movies[['movie_id',
                 'title',
                 'overview',
                 'genres',
                 'keywords',
                 'cast',
                 'crew']]


## 3.3 Handle Missing Values

In [ ]:
# drop missing values
movies.dropna(inplace=True)


In [ ]:
# check if missing values were removed
movies.isnull().sum()


In [ ]:
# No duplicated rows
movies.duplicated().sum()

## 3.4 Feature Engineering & Natural Language Processing

Main steps include:
- converting JSON-like columns into usable lists
- extracting important features (genres, keywords, cast, crew)
- creating a unified "tags" column
- applying text preprocessing (lowercasing, tokenization, stemming)

## 3.4.1 Convert JSON-like columns into usable format & feature extraction

In [ ]:
import ast
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Convert stringified JSON in genres & keywords into Python objects.
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

# Apply the function on the genres column
movies['genres'] = movies['genres'].apply(convert)

# Apply the function on the keywords column
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
# Clean "cast" column
def convert_cast(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

# Apply the fuction on cast
movies['cast'] = movies['cast'].apply(convert_cast)


In [ ]:
# clean "crew" column
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

# Apply the function on crew
movies['crew'] = movies['crew'].apply(fetch_director)


## 3.4.2 Tokenization
* To split longsentences into individual words

In [ ]:
# Converts overview text into tokens
movies['overview'] = movies['overview'].apply(lambda x: x.split())

## 3.4.3 Removal of extra spaces

In [ ]:
# Remove extra spaces for consistency in text processing
def remove_spaces(L):
    return [i.replace(" ", "") for i in L]

# Apply to the columns
movies['genres'] = movies['genres'].apply(remove_spaces)
movies['keywords'] = movies['keywords'].apply(remove_spaces)
movies['cast'] = movies['cast'].apply(remove_spaces)
movies['crew'] = movies['crew'].apply(remove_spaces)

## 3.4.4 Create unified “Tags” column

* The tags column combine all important text-based features into a single column called "tags".

In [ ]:
# Combine all the columns into one list per movie
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Create Final DataFrame to only essential columns for recommendation.
new_df = movies[['movie_id', 'title', 'tags']]

# Convert list of words into a single string so it can be processed by NLP models
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

## 3.4.5 Lowercasing

In [ ]:
# Normalize text into lowercase to ensure consistency in vectorization.
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

## 3.4.6 Stemming

* To reduce words to their root form. 

In [ ]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

# stem function
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

# Apply stemming
new_df['tags'] = new_df['tags'].apply(stem)

# 4.0 MODELING

## 4.1 CountVectorizer as the baseline recommender

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')

# Convert text to vectors
vectors = cv.fit_transform(new_df['tags']).toarray()

# Compute similarity matrix
similarity = cosine_similarity(vectors)

## 4.2 TF-IDF as improved Model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

# Convert text to vectors
vectors_tfidf = tfidf.fit_transform(new_df['tags']).toarray()

# Compute similarity matrix
similarity_tfidf = cosine_similarity(vectors_tfidf)

* Vectorization is applied to transform texts into numerical vectors for machine learning processing.
* The model limits the unique words to the 5,000 most frequent words while removing stop words (common words that do not add meaningful information).
* Cosine similarity is used to measure similarity between movies, enabling the recommendation of movies with related content.

## 4.3 Recommendation Function

In [ ]:
def recommend(movie, similarity_matrix):
    movie_index = new_df[new_df['title'] == movie].index[0]
    
    distances = similarity_matrix[movie_index]
    
    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:10]
    
    for i in movie_list:
        print(new_df.iloc[i[0]].title)

In [ ]:
# Test Recommendations
recommend('Meet Dave', similarity_tfidf)

In [ ]:
# Saving Model Files for deployment
import pickle

pickle.dump(new_df, open('movies.pkl', 'wb'))
pickle.dump(similarity_tfidf, open('similarity.pkl', 'wb'))

# 5.0 Conclusion

* A content-based movie recommendation system was successfully built using NLP techniques.
* The system processes movie metadata, transforms it into numerical vectors and computes similarity using cosine similarity to generate recommendations.
